In [ ]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split\test_df_prepared.parquet", engine="pyarrow")

In [ ]:
cols_to_drop = ["flow_id", "timestamp", "src_ip", "src_port", "dst_ip", "dst_port"]
train_df = train_df.drop(columns=cols_to_drop)
valid_df = valid_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)

KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, classification_report

X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

# Thử các giá trị k khác nhau
k_values = [3, 5, 7, 9, 11, 15, 21]

best_k = None
best_f1 = -1
best_model = None

print("Đang huấn luyện phân lớp KNN và tìm 'k' tốt nhất dựa trên Validation Macro F1...")

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train, y_train)
    
    y_valid_pred = knn.predict(X_valid)
    
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"k = {k:2d} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_k = k
        best_model = knn

print(f"\n=> K tốt nhất tìm được là k = {best_k} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test
print("\n--- Đánh giá Model Tốt nhất (k={}) trên tập TEST ---".format(best_k))
y_test_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))


SVM

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Linear SVM (bằng SGDClassifier) và tìm tham số 'alpha' tốt nhất...")
start_time = time.time()

# Thử các giá trị alpha
alpha_values = [1e-3, 1e-4, 1e-5, 1e-6]

best_alpha = None
best_f1 = -1
best_svm_model = None

for alpha in alpha_values:
    svm_model = SGDClassifier(
        loss='hinge', 
        penalty='l2', 
        alpha=alpha, 
        max_iter=1000, 
        tol=1e-3, 
        n_jobs=-1,    
        random_state=42
    )

    # Huấn luyện mô hình
    svm_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính Macro F1
    y_valid_pred = svm_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_alpha = alpha
        best_svm_model = svm_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test
print("\n--- Đánh giá Model SVM Tốt nhất (alpha={}) trên tập TEST ---".format(best_alpha))
y_test_pred = best_svm_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))


Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Random Forest và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử trên các max_depth khác nhau
max_depth_values = [10, 15, 20, 25, 30]

best_depth = None
best_f1 = -1
best_rf_model = None

for depth in max_depth_values:
    rf_model = RandomForestClassifier(
        n_estimators=100, 
        max_depth=depth,
        n_jobs=-1, 
        random_state=42
    )

    # Huấn luyện mô hình
    rf_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính điểm Macro F1
    y_valid_pred = rf_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_rf_model = rf_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Random Forest Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_rf_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))


Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Decision Tree và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử nghiệm các giá trị max_depth khác nhau (giống cấu hình tuning trên Random Forest)
max_depth_values = [10, 15, 20, 25, 30]

best_depth = None
best_f1 = -1
best_dt_model = None

for depth in max_depth_values:
    # Dùng entropy để tính information gain và thay đổi độ sâu max_depth
    dt_model = DecisionTreeClassifier(
        criterion='entropy',
        max_depth=depth,
        random_state=42
    )

    # Huấn luyện mô hình
    dt_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính Macro F1
    y_valid_pred = dt_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_dt_model = dt_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_dt_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))


Logistic Regression

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Logistic Regression và tìm 'alpha' & 'penalty' tốt nhất...")
start_time = time.time()

# Thử nghiệm các alpa và penalt l1, l2
alpha_values = [1e-3, 1e-4, 1e-5, 1e-6]
penalty_values = ['l1', 'l2']

best_alpha = None
best_penalty = None
best_f1 = -1
best_logreg_model = None

for penalty in penalty_values:
    for alpha in alpha_values:
        # Dùng hàm loss là log_loss
        logreg_model = SGDClassifier(
            loss='log_loss',  
            penalty=penalty,
            alpha=alpha, 
            max_iter=1000,
            tol=1e-3,
            n_jobs=-1,
            random_state=42
        )

        # Huấn luyện mô hình
        logreg_model.fit(X_train, y_train)
        
        # Dự đoán trên tập validation và tính Macro F1
        y_valid_pred = logreg_model.predict(X_valid)
        current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
        
        print(f"penalty = {penalty:2s}, alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
        
        # Chọn model tốt nhất dựa trên Macro F1
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_alpha = alpha
            best_penalty = penalty
            best_logreg_model = logreg_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: penalty = {best_penalty}, alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print(f"\n--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty={best_penalty}, alpha={best_alpha}) trên tập TEST ---")
y_test_pred = best_logreg_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))
